# ERA5 from the cloud as an MdGdalSource dataset

This notebook registers ERA5 pressure-level data straight from the public NCAR bucket
`https://nsf-ncar-era5.s3.amazonaws.com/`. Nothing is downloaded: the server opens each file with
GDAL's `/vsicurl/` and reads only the bytes a query needs.

The file is a netCDF with dimensions `(time, level, latitude, longitude)` and variable `PV`
(potential vorticity). `MdGdalSource` reads it as a raster with:

* **time** as the z axis (`Time`): one slice per hour,
* **one pressure level** selected with `Fixed` (index 21 is 500 hPa in this file),
* **latitude** descending, which the server flips to north-up on read,
* **longitude** stored as `0..360` and kept as they are (`wrap=False`).

This notebook needs a running Geo Engine server with network access to the bucket. It is not
run in CI.

In [ ]:
import datetime as dt

import geoengine_api_client as geoc
import numpy as np

import geoengine as ge

ge.initialize("http://localhost:3030/api", ("admin@localhost", "adminadmin"))

BUCKET = "https://nsf-ncar-era5.s3.amazonaws.com"
PREFIX = "e5.oper.an.pl/200001"
DAYS = [1, 2, 3]  # 2000-01-01 .. 2000-01-03, one file per day

## Describe the array

The stored grid is the file's own grid: 1440 x 721 cells of 0.25 degrees, with the origin at the
north-west pixel edge (`-0.125, 90.125`) and a negative y pixel size. Longitudes stay 0..360:
0 is a pixel *centre* here, so the origin is half a 0.25 degree pixel off the -180 lattice that
`wrap=True` needs, and the wrap is rejected.

In [ ]:
WIDTH, HEIGHT = 1440, 721
STORED_GEO_TRANSFORM = geoc.GeoTransform(
    origin_coordinate=geoc.Coordinate2D(x=-0.125, y=90.125),
    x_pixel_size=0.25,
    y_pixel_size=-0.25,
)
HOUR_MS = 3_600_000
FIRST_HOUR_MS = 946684800000  # 2000-01-01T00:00:00Z

result_descriptor = geoc.RasterResultDescriptor(
    data_type="F32",
    spatial_reference="EPSG:4326",
    time=geoc.TimeDescriptor(
        bounds=geoc.TimeInterval(start=FIRST_HOUR_MS, end=FIRST_HOUR_MS + 3 * 24 * HOUR_MS),
        dimension=geoc.TimeDimension(
            actual_instance=geoc.Regular(
                origin=FIRST_HOUR_MS,
                step=geoc.TimeStep(granularity="hours", step=1),
                type="regular",
            )
        ),
    ),
    spatial_grid=geoc.SpatialGridDescriptor(
        spatialGrid=geoc.SpatialGridDefinition(
            geoTransform=STORED_GEO_TRANSFORM,
            gridBounds=geoc.GridBoundingBox2D(
                topLeftIdx=geoc.GridIdx2D(xIdx=0, yIdx=0),
                bottomRightIdx=geoc.GridIdx2D(xIdx=WIDTH - 1, yIdx=HEIGHT - 1),
            ),
        ),
        descriptor="source",
    ),
    measurement=geoc.Measurement(actual_instance=geoc.UnitlessMeasurement(type="unitless")),
    bands=[
        geoc.RasterBandDescriptor(
            name="PV",
            measurement=geoc.Measurement(actual_instance=geoc.UnitlessMeasurement(type="unitless")),
        )
    ],
)

md = geoc.GdalMdMetaData(
    type="GdalMdMetaData",
    result_descriptor=result_descriptor,
    variables=[
        geoc.MdVariable(
            name="PV",
            axisMapping=[
                geoc.MdAxisMapping(actual_instance=geoc.MdAxisMappingOneOf2(role="time")),
                # index 21 of the file's `level` dimension is 500 hPa
                geoc.MdAxisMapping(actual_instance=geoc.MdAxisMappingOneOf3(role="fixed", index=21)),
                geoc.MdAxisMapping(actual_instance=geoc.MdAxisMappingOneOf1(role="y")),
                geoc.MdAxisMapping(actual_instance=geoc.MdAxisMappingOneOf(role="x")),
            ],
        )
    ],
    wrap=False,
    cacheTtl=None,
)

meta_data = geoc.MetaDataDefinition(actual_instance=md)
print(f"variables={[v.name for v in md.variables]}, wrap={md.wrap}")

## Describe the files

One row per daily file. Each file holds 24 hourly slices, so `timeSteps` has 24 intervals and the
file path is the `/vsicurl/` URL. The dataset is registered as `external`, so the server opens the
URL itself.

In [ ]:
def file_name(day: int) -> str:
    stamp = f"200001{day:02d}"
    return f"e5.oper.an.pl.128_060_pv.ll025sc.{stamp}00_{stamp}23.nc"


spatial_partition = geoc.SpatialPartition2D(
    upperLeftCoordinate=geoc.Coordinate2D(x=-0.125, y=90.125),
    lowerRightCoordinate=geoc.Coordinate2D(x=359.875, y=-90.125),
)

tiles = []
for day in DAYS:
    start = FIRST_HOUR_MS + (day - 1) * 24 * HOUR_MS
    time_steps = [geoc.TimeInterval(start=start + h * HOUR_MS, end=start + (h + 1) * HOUR_MS) for h in range(24)]
    tiles.append(
        geoc.AddDatasetMdTile(
            spatial_partition=spatial_partition,
            band=0,
            z_index=len(tiles),
            array_name="PV",
            arrayGroup=None,
            time_descriptor=geoc.TimeDescriptor(
                bounds=geoc.TimeInterval(start=time_steps[0].start, end=time_steps[-1].end),
                dimension=geoc.TimeDimension(
                    actual_instance=geoc.Regular(
                        origin=time_steps[0].start,
                        step=geoc.TimeStep(granularity="hours", step=1),
                        type="regular",
                    )
                ),
            ),
            time_steps=time_steps,
            params=geoc.GdalDatasetParameters(
                file_path=f"/vsicurl/{BUCKET}/{PREFIX}/{file_name(day)}",
                rasterband_channel=1,
                geo_transform=STORED_GEO_TRANSFORM,
                width=WIDTH,
                height=HEIGHT,
                file_not_found_handling="NoData",
                noDataValue=None,
            ),
        )
    )
print(f"{len(tiles)} files, {len(tiles[0].time_steps)} hours each")

## Register

The dataset is public for reading, like the other example datasets.

In [ ]:
dataset = ge.add_or_replace_dataset_with_permissions(
    "external",
    ge.AddDatasetProperties(
        name="era5_pv_cloud",
        display_name="ERA5 PV at 500 hPa (cloud)",
        description="Hourly ERA5 potential vorticity at 500 hPa, read from the NCAR bucket over /vsicurl/.",
        source_operator="MdGdalSource",
        symbology=ge.RasterSymbology(
            raster_colorizer=ge.SingleBandRasterColorizer(
                band=0,
                band_colorizer=ge.Colorizer.linear_with_mpl_cmap(color_map="RdBu", min_max=(-0.03, 0.03), n_steps=10),
            ),
        ),
    ),
    meta_data,
    permission_tuples=[
        (ge.permissions.ANONYMOUS_USER_ROLE_ID, ge.Permission.READ),
        (ge.permissions.REGISTERED_USER_ROLE_ID, ge.Permission.READ),
    ],
    replace_existing=True,
    timeout=600,
)

ge.add_md_dataset_tiles(dataset, tiles, timeout=600)
print("registered:", dataset)

## Read a window

A small window over Europe (0° to 30° East, inside the file's 0..360 longitudes) for three hours of
the first day. The check is loose on purpose: it
confirms the values are finite and inside the file's range (potential vorticity at 500 hPa is of order 1e-6 K m² kg⁻¹ s⁻¹), which fails if
the axis mapping, the level or the flip is wrong.

In [ ]:
workflow = ge.register_workflow(workflow=ge.workflow_builder.operators.MdGdalSource(dataset=dataset))

window = ge.TimeInterval(
    start=dt.datetime(2000, 1, 1, 6, tzinfo=dt.timezone.utc),
    end=dt.datetime(2000, 1, 1, 9, tzinfo=dt.timezone.utc),
)
bbox = ge.QueryRectangle(ge.BoundingBox2D(0.0, 35.0, 30.0, 60.0), window)

tiles_read = [tile async for tile in workflow.raster_stream(bbox, open_timeout=600)]
print(f"{len(tiles_read)} tiles")

values = np.concatenate([t.to_numpy_data_array().ravel() for t in tiles_read])
valid = values[np.isfinite(values)]
assert valid.size > 0
assert np.all(np.isfinite(valid)) and np.all(np.abs(valid) < 0.1), (valid.min(), valid.max())
print(f"valid cells: {valid.size}, range [{valid.min():.3e} .. {valid.max():.3e}]")

## Clean up

In [ ]:
ge.datasets.delete_dataset(dataset)
ge.reset()
print("deleted:", dataset)